# Import libraries required for finetuning

In [ ]:
import argparse
import os
import sys
import time
import torch
import mlflow
import pandas as pd

from pathlib import Path
from datasets import load_from_disk
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
    AutoProcessor
)
from functools import partial

# Set variables that will be used e.g. in model loading, finetuning and loading data

In [ ]:
SLURM_JOB_ACCOUNT = os.getenv("SLURM_JOB_ACCOUNT") #modify
USER = os.getenv("SLURM_JOB_USER") #modify

In [ ]:
input_model = "Qwen/Qwen3-4B-Instruct-2507"
output_path = f"/scratch/{SLURM_JOB_ACCOUNT}/{USER}/health_case/ft_model"
model_output_name = f"{input_model}_finetuned"
json_file = f"/scratch/{SLURM_JOB_ACCOUNT}/data/structured_notes.json"
batch_size = 2
cache_dir = f"/scratch/{SLURM_JOB_ACCOUNT}/hf-cache/hub"
max_tokens = 2048
num_workers = int(os.getenv("SLURM_CPUS_PER_TASK"))

output_model_dir = os.path.join(output_path, model_output_name)
merged_output_dir = os.path.join(
    output_path,
    f"{model_output_name}_merged"
)

# Define a function that transforms conversations and structured notes to tokenized output

In [ ]:
def preprocess(examples, tokenizer, max_tokens=2048):
    """
    Convert chat examples into tokenized causal-LM training examples.

    Safeguards:
    - Skip samples where the assistant response is completely truncated.
    - Ensure labels and input_ids always have identical lengths.
    - Ensure at least one non-masked label token exists.
    """

    input_ids_list = []
    attention_mask_list = []
    labels_list = []

    skipped_no_response = 0
    skipped_empty_labels = 0

    for conversation, structured_note in zip(
        examples["conversation"],
        examples["structured_note"],
    ):

        # Build chat messages
        messages = [
            {
                "role": "system",
                "content": """You are a medical clinical documentation assistant. 
You task is to convert a dialogue between a doctor and patient into a structured clinical note in the following output format:
REASON FOR VISIT:
<Brief summary of why the patient is seeking care>
PATIENT DETAILS AND HISTORY:
<Age, gender, relevant demographics, relevant past medical history, conditions, medications, surgeries, lifestyle factors>
CURRENT STATUS:
<Current symptoms, findings, vitals, clinical observations>
TREATMENTS/ACTIONS:
<Medications prescribed, procedures performed, advice given>
FOLLOW-UP PLAN:
<Next steps, monitoring, referrals, timelines. Follow-up plan should not include "future" details that are mentioned in the note, but rather should infer what the next steps would be based on the found future details.>
"""},
            {"role": "user", "content": conversation},
            {"role": "assistant", "content": structured_note},
        ]

        # Apply chat template — tokenize full conversation
        full_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        prompt_text = tokenizer.apply_chat_template(
            messages[:-1],
            tokenize=False,
            add_generation_prompt=True,
        )

        full_enc = tokenizer(
            full_text,
            truncation=True,
            max_length=max_tokens,
            padding=False,
            add_special_tokens=False,
        )

        prompt_enc = tokenizer(
            prompt_text,
            truncation=True,
            max_length=max_tokens,
            padding=False,
            add_special_tokens=False,
        )

        input_ids = full_enc["input_ids"]
        attention_mask = full_enc["attention_mask"]

        # Safety: prompt length cannot exceed actual sequence length
        prompt_len = min(len(prompt_enc["input_ids"]), len(input_ids))

        # Assistant response completely truncated
        if prompt_len >= len(input_ids):
            skipped_no_response += 1
            continue

        labels = [-100] * prompt_len + input_ids[prompt_len:]

        # Safety check
        if len(labels) != len(input_ids):
            raise ValueError(
                f"Label length mismatch: labels={len(labels)}, "
                f"input_ids={len(input_ids)}"
            )

        valid_label_count = sum(label != -100 for label in labels)

        if valid_label_count == 0:
            skipped_empty_labels += 1
            continue

        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)

    print(
        f"Skipped {skipped_no_response} samples "
        f"(assistant response truncated)"
    )
    print(
        f"Skipped {skipped_empty_labels} samples "
        f"(no valid label tokens)"
    )

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "labels": labels_list,
    }

# Load tokenizer and models

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(input_model, use_fast=True, cache_dir=cache_dir)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    input_model,
    torch_dtype=torch.bfloat16,
    device_map=device,
    cache_dir=cache_dir
)

In [ ]:
peft_config = LoraConfig(
    lora_alpha=8,
    lora_dropout=0.05,
    r=16,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

In [ ]:
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Load and setup the data

In this notebook, we use the validation split (3k conversations) of the bigger dataset (30k conversations). This validation dataset will be used to conduct a smaller scale finetuning suitable in this notebook environment.

In [ ]:
## DATA
dataset = load_from_disk(f"{Path(json_file).parent / 'raw_val'}")
split = dataset.train_test_split(test_size=0.1, seed=42)

raw_train = split["train"] #.select(list(range(0, 2000)))
raw_val = split["test"]

print(f"  Train size: {len(raw_train)}")
print(f"  Val size:   {len(raw_val)}")

We will also quickly inspect at some of the examples of the data. Conversation and structured_notes will be used as the finetuning material.

In [ ]:
dataset_df = pd.DataFrame(raw_val)

In [ ]:
dataset_df

In [ ]:
preprocess_fn = partial(preprocess, tokenizer=tokenizer, max_tokens=max_tokens)

In [ ]:
tokenized_train = raw_train.map(
    preprocess_fn,
    batched=True,
    remove_columns=raw_train.column_names,
    num_proc=num_workers,
)

In [ ]:
tokenized_val = raw_val.map(
    preprocess_fn,
    batched=True,
    remove_columns=raw_val.column_names,
    num_proc=num_workers,
)

# Setting up the HuggingFace Trainer with required arguments for finetuning

In [ ]:
training_args = TrainingArguments(
    disable_tqdm=False,
    output_dir=output_model_dir,
    save_strategy="steps",
    save_steps=150,
    save_total_limit=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    bf16=True,
    load_best_model_at_end=True,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size*2,
    dataloader_num_workers=num_workers,
    ddp_find_unused_parameters=True,
    dataloader_pin_memory=True,
    save_safetensors=True,
    metric_for_best_model="eval_loss",
    eval_strategy="steps",
    eval_steps=150,
    num_train_epochs=1,
    report_to=["mlflow"],
    logging_steps=50,
    logging_strategy="steps",
    run_name=f"{model_output_name}_{os.environ.get('SLURM_JOB_ID', 'local')}",
)

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Start the finetuning process

In [ ]:
start_train = time.time()

trainer.train()

stop_train = time.time()

In [ ]:
elapsed = stop_train - start_train
hours   = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = int(elapsed % 60)
print(f"Training took: {hours}h {minutes}m {seconds}s")

# Save the finetuned model for testing in the [inference notebook](./inference_nb.ipynb)

In [ ]:
merged_model = model.merge_and_unload()

merged_model.save_pretrained(
    merged_output_dir,
    safe_serialization=False
)

tokenizer.save_pretrained(merged_output_dir)
processor.save_pretrained(merged_output_dir)

# IMPORTANT NOTE

Once you have reached this far, be adviced to restart the kernel (the notebook environment), so that the GPU resources are freed from the finetuning process. This is required to be able to try inference in the next notebook.

In [ ]:
from IPython import get_ipython

get_ipython().kernel.do_shutdown(restart=True)